# Smart City NYC — Processing City Data with Spark DataFrames

**Case Study 2 · Unit 5.2 DataFrames · Tools: PySpark, Pandas**

| Name | Roll No |
|---|---|
| Varun Murugan | 124A8118 |
| Rankin Yanu | 124A8124 |
| Navin Nadar | 225A8134 |
| Abhay Hanchate | 225A8129 |

## What this notebook does (in simple words)

A *smart city* uses data from vehicles, citizens, sensors and roads to run the city better.
We act as New York City's data team:

1. **Collect** six real city datasets, from files and from web APIs (one of them is live).
2. **Load** them into Spark DataFrames and **clean** them.
3. **Answer 10 questions** about mobility, citizen complaints, air quality and live traffic.
4. **Speed up** the processing with 5 optimizations and **measure** each one.
5. **Compare** Spark with Pandas, and 1 CPU core with many.

## The data

| Area | Data | Format | Size |
|---|---|---|---|
| 🚕 Mobility | NYC Yellow Taxi trips, January 2024 | Parquet file | ~3 million rows |
| 🚕 Mobility | Taxi zone names | CSV file | 265 rows |
| 🏛️ Citizen services | 311 complaints, January 2024 | CSV from the NYC Open Data API | ~250,000 rows |
| 🌫️ Environment | Hourly air quality (PM2.5, NO₂, CO) | JSON from the Open-Meteo API | 744 rows |
| 🌦️ Environment | Hourly weather (temperature, rain) | JSON from the Open-Meteo API | 744 rows |
| 🚦 Live traffic | Road speeds from city sensors, **right now** | JSON from the NYC DOT live API | latest readings |

## Five words to know before you start

| Word | Meaning |
|---|---|
| **DataFrame** | A table with named, typed columns. In Spark it is split into pieces and processed in parallel. |
| **Partition** | One piece of a DataFrame. Each partition is processed by one CPU core at a time. |
| **Transformation** | A step like `filter`, `select`, `groupBy`, `join`. It only *plans* work; nothing runs yet. |
| **Action** | A request for a result, like `count`, `show`, `toPandas`. This is when Spark actually runs the plan. |
| **Shuffle** | Moving rows between partitions so that matching rows meet (needed by `groupBy`, `join`, sorting). The slowest part of most jobs. |

**How to run:** in Google Colab click *Runtime → Run all*. Everything is installed and downloaded automatically
(about 5 minutes). Cells marked 📸 tell you when to take a screenshot for the report.

**The 7 steps:** 1 Set up Spark → 2 Load the data → 3 See how Spark splits the work → 4 Clean the data and answer
10 questions → 5 Make it faster → 6 Save the results → 7 Analyze performance

## Step 1 — Set up Spark

**What happens here:** we install PySpark if it is missing, then start a **SparkSession**, the entry point to
everything in Spark.

**Key idea:** we run Spark in **local mode** (`local[*]`). Spark runs on this one machine and uses **every CPU core**
as a parallel worker. On a real cluster the same code works unchanged; only the `master` setting would change.

In [ ]:
# Install PySpark / Java only if missing (Colab usually has both)
import importlib.util, os, re, shutil, subprocess, sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)
import pyspark

def java_major():
    if shutil.which("java") is None:
        return 0
    out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    found = re.search(r'version "(?:1\.)?(\d+)', out)
    return int(found.group(1)) if found else 0

needed = 17 if int(pyspark.__version__.split(".")[0]) >= 4 else 8   # Spark 4 needs Java 17+
if java_major() < needed:
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
print(f"PySpark {pyspark.__version__} with Java {java_major()} is ready")

In [ ]:
import os, time, statistics, urllib.request
from functools import reduce

import pandas as pd
import matplotlib.pyplot as plt
import pyspark
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, StringType

def create_spark(master="local[*]"):
    return (SparkSession.builder
            .appName("NYC-Taxi-DataFrames-CaseStudy")
            .master(master)
            .config("spark.driver.memory", "4g")
            .config("spark.sql.adaptive.enabled", "true")   # Adaptive Query Execution (default in Spark 3.2+)
            .getOrCreate())

spark = create_spark()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version       :", spark.version)
print("PySpark version     :", pyspark.__version__)
print("Pandas version      :", pd.__version__)
print("Master              :", spark.sparkContext.master)
print("Parallel cores      :", spark.sparkContext.defaultParallelism)
print("Spark UI            :", spark.sparkContext.uiWebUrl)

> 📸 **SCREENSHOT 1** — the output above (Spark version, master, number of cores).

### Open the Spark UI (Spark's live dashboard)

The Spark UI shows every job, stage and task Spark runs, what is cached, and how each query was executed.
Steps 3 and 5 ask for screenshots of its tabs.

**Where to open it:**
* **In Google Colab:** click the **"Open Spark UI"** link that the next cell prints.
  Do **not** type `localhost:4040` in your own browser. Spark runs on Google's computer, not on your laptop,
  so your laptop has nothing at that address ("refused to connect").
* **On your own laptop (Jupyter):** open the `http://localhost:...` address the next cell prints.
* **If the Colab link shows an error or a blank page:** run the backup cell after it. It creates a temporary
  public link to the Spark UI.

In [ ]:
UI_PORT = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[-1])   # usually 4040 (4041 if 4040 was busy)

try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    output.serve_kernel_port_as_window(UI_PORT, path="/jobs/", anchor_text="👉 Open Spark UI")
else:
    print(f"Open in your browser: http://localhost:{UI_PORT}/jobs/")

#### Backup: open the Spark UI through a public link (only if the link above does not work)

Set `USE_PUBLIC_LINK = True` and run the cell. It downloads Cloudflare's free tunnel tool (no account needed)
and prints a link like `https://something.trycloudflare.com/jobs/` that opens the Spark UI in any browser.
The link works only while this notebook is running. Anyone who has the link can view the Spark UI, so don't share it.

In [ ]:
USE_PUBLIC_LINK = False   # change to True if the "Open Spark UI" link above does not work

if USE_PUBLIC_LINK:
    import re
    if not os.path.exists("cloudflared"):
        subprocess.run("wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/"
                       "cloudflared-linux-amd64 && chmod +x cloudflared", shell=True, check=True)
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{UI_PORT}"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(200):                                    # the tool prints the link within a few seconds
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel.stdout.readline())
        if found:
            print("Spark UI public link:", found.group(0) + "/jobs/")
            break
    else:
        print("Could not create the link; re-run this cell.")
else:
    print("Backup link not needed. Set USE_PUBLIC_LINK = True only if the Spark UI link above does not open.")

## Step 2 — Load the data (files and web APIs)

**What happens here:** the next cell downloads all six sources into `data/raw/`. Files that already exist are
skipped, except live traffic, which is always fetched fresh.

| Source | How we get it |
|---|---|
| Taxi trips, zones | Direct file download |
| 311 complaints | NYC Open Data API, in pages of 100,000 rows (one CSV file per page) |
| Air quality, weather | Open-Meteo API, returns JSON |
| Live traffic | NYC DOT real-time API, returns JSON. If it is unreachable we reuse the last saved snapshot. |

Then each source is read into its own Spark DataFrame (2a–2e).

In [ ]:
import json
from urllib.parse import urlencode

RAW, PROCESSED = "data/raw", "data/processed"
CHARTS, RESULTS = "outputs/charts", "outputs/results"
for d in (RAW, PROCESSED, CHARTS, RESULTS):
    os.makedirs(d, exist_ok=True)

TRIPS_PATH   = f"{RAW}/yellow_tripdata_2024-01.parquet"
ZONES_PATH   = f"{RAW}/taxi_zone_lookup.csv"
WEATHER_PATH = f"{RAW}/weather_nyc_2024-01.json"
AIR_PATH     = f"{RAW}/air_quality_nyc_2024-01.json"
C311_DIR     = f"{RAW}/nyc_311_2024-01"
TRAFFIC_DIR  = f"{RAW}/traffic_live"

OPEN_METEO = "latitude=40.71&longitude=-74.01&start_date=2024-01-01&end_date=2024-01-31&timezone=America/New_York"
FILES = {
    TRIPS_PATH:   "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
    ZONES_PATH:   "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv",
    WEATHER_PATH: f"https://archive-api.open-meteo.com/v1/archive?{OPEN_METEO}&hourly=temperature_2m,precipitation,rain",
    AIR_PATH:     f"https://air-quality-api.open-meteo.com/v1/air-quality?{OPEN_METEO}"
                  "&hourly=pm2_5,nitrogen_dioxide,carbon_monoxide",
}

def http_get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=300) as resp:
        return resp.read()

for path, url in FILES.items():
    if not os.path.exists(path):
        print("Downloading", path)
        with open(path, "wb") as f:
            f.write(http_get(url))

# 311 service requests: NYC Open Data (Socrata) API, paged CSV download
SOCRATA_311 = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv"
PAGE = 100_000
if not os.path.exists(f"{C311_DIR}/_COMPLETE"):
    os.makedirs(C311_DIR, exist_ok=True)
    offset, page_no = 0, 0
    while True:
        query = urlencode({
            "$select": "unique_key,created_date,closed_date,agency,complaint_type,descriptor,"
                       "borough,incident_zip,latitude,longitude,status",
            "$where":  "created_date between '2024-01-01T00:00:00' and '2024-01-31T23:59:59'",
            "$order":  "unique_key",
            "$limit":  PAGE, "$offset": offset})
        body = http_get(f"{SOCRATA_311}?{query}")
        rows = body.count(b"\n") - 1                       # minus the header line
        with open(f"{C311_DIR}/part_{page_no:02d}.csv", "wb") as f:
            f.write(body)
        print(f"311 API page {page_no}: {rows:,} rows")
        if rows < PAGE:
            break
        offset, page_no = offset + PAGE, page_no + 1
    open(f"{C311_DIR}/_COMPLETE", "w").close()

# Live traffic speeds: NYC DOT real-time API, fetched fresh on every run
os.makedirs(TRAFFIC_DIR, exist_ok=True)
try:
    query = urlencode({"$select": "id,speed,travel_time,status,data_as_of,link_id,borough,link_name",
                       "$order": "data_as_of DESC", "$limit": 5000})
    body = http_get(f"https://data.cityofnewyork.us/resource/i4gi-tjb9.json?{query}")
    TRAFFIC_PATH = f"{TRAFFIC_DIR}/traffic_{time.strftime('%Y%m%d_%H%M%S')}.json"
    with open(TRAFFIC_PATH, "wb") as f:
        f.write(body)
    print("Fetched live traffic snapshot ->", TRAFFIC_PATH)
except Exception as err:                                   # offline: reuse the latest snapshot on disk
    saved = sorted(f for f in os.listdir(TRAFFIC_DIR) if f.endswith(".json"))
    if not saved:
        raise RuntimeError("Live traffic API unreachable and no saved snapshot; re-run this cell") from err
    TRAFFIC_PATH = f"{TRAFFIC_DIR}/{saved[-1]}"
    print(f"Live traffic API unreachable ({err.__class__.__name__}); using saved snapshot {TRAFFIC_PATH}")

def size_mb(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1e6
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(path) for f in fs) / 1e6

print()
for path in (TRIPS_PATH, ZONES_PATH, C311_DIR, AIR_PATH, WEATHER_PATH, TRAFFIC_PATH):
    print(f"{path:55s} {size_mb(path):8.2f} MB")

### 2a. 🚕 Taxi trips (Parquet file)

Parquet stores the column names and types inside the file, so Spark needs no schema from us.
We print the schema (the list of columns and types), count the rows, and show the first 5.

In [ ]:
trips_raw = spark.read.parquet(TRIPS_PATH)
trips_raw.printSchema()
print(f"Rows: {trips_raw.count():,}   Columns: {len(trips_raw.columns)}")
trips_raw.show(5)

### 2b. 🚕 Taxi zones (CSV file, with a schema we define)

A CSV file is plain text with no types, so we tell Spark the column names and types ourselves (`StructType`).
This is **faster** than `inferSchema=True`, which reads the file an extra time to guess the types, and **safer**,
because a guess can be wrong.

In [ ]:
zone_schema = StructType([
    StructField("LocationID",   IntegerType()),
    StructField("Borough",      StringType()),
    StructField("Zone",         StringType()),
    StructField("service_zone", StringType()),
])
zones = spark.read.csv(ZONES_PATH, header=True, schema=zone_schema)
print(f"Rows: {zones.count():,}")
zones.show(5, truncate=False)

### 2c. 🏛️ 311 complaints (a folder of CSV files → one DataFrame)

The API gave us the data as several CSV pages. Spark reads **the whole folder at once**, each file in parallel,
as one table. Dates are read as text for now; Step 4 converts them.

In [ ]:
c311_schema = StructType([
    StructField("unique_key",     LongType()),
    StructField("created_date",   StringType()),
    StructField("closed_date",    StringType()),
    StructField("agency",         StringType()),
    StructField("complaint_type", StringType()),
    StructField("descriptor",     StringType()),
    StructField("borough",        StringType()),
    StructField("incident_zip",   StringType()),
    StructField("latitude",       DoubleType()),
    StructField("longitude",      DoubleType()),
    StructField("status",         StringType()),
])
c311_raw = (spark.read.option("multiLine", True).option("escape", '"')
                 .csv(C311_DIR, header=True, schema=c311_schema))
c311_raw.printSchema()
print(f"Rows: {c311_raw.count():,}   Files: {len(c311_raw.inputFiles())}")
c311_raw.show(5, truncate=30)

### 2d. 🌫️🌦️ Air quality and weather (nested JSON → flat table)

The APIs return the hourly values as lists side by side:
`"time": [00:00, 01:00, …]`, `"nitrogen_dioxide": [24.8, 20.8, …]`.

* `arrays_zip` pairs the lists element by element: (00:00, 24.8), (01:00, 20.8), …
* `explode` turns that one list into **one row per hour**.

The result is a normal table: `hour_ts | pm2_5 | nitrogen_dioxide | carbon_monoxide`.

In [ ]:
def hourly_api_json(path, fields):
    """Flatten an Open-Meteo JSON response into one row per hour."""
    return (spark.read.option("multiLine", True).json(path)
                 .select("hourly.*")
                 .select(F.explode(F.arrays_zip("time", *fields)).alias("h"))
                 .select(F.to_timestamp("h.time", "yyyy-MM-dd'T'HH:mm").alias("hour_ts"),
                         *[F.col(f"h.{f}").alias(f) for f in fields]))

spark.read.option("multiLine", True).json(AIR_PATH).printSchema()

air = hourly_api_json(AIR_PATH, ["pm2_5", "nitrogen_dioxide", "carbon_monoxide"])
weather = (hourly_api_json(WEATHER_PATH, ["temperature_2m", "precipitation", "rain"])
           .withColumnRenamed("temperature_2m", "temp_c")
           .withColumnRenamed("precipitation", "precip_mm"))
print(f"Air quality rows: {air.count()}   Weather rows: {weather.count()}")
air.show(5)
weather.show(5)

### 2e. 🚦 Live traffic speeds (JSON from a real-time API)

This file was downloaded a moment ago, so it shows how fast traffic is moving on NYC roads **right now**.
Each row is one sensor reading for one road. All values arrive as text; Step 4 converts them to numbers.

In [ ]:
traffic_raw = spark.read.option("multiLine", True).json(TRAFFIC_PATH)
traffic_raw.printSchema()
print(f"Sensor readings in this snapshot: {traffic_raw.count():,}")
traffic_raw.orderBy(F.desc("data_as_of")).show(5, truncate=40)

> 📸 **SCREENSHOT 2** — the download sizes, and the schemas and row counts of all datasets (cells above).

## Step 3 — See how Spark splits the work (distributed processing)

**What happens here:** we look at how Spark divides the taxi data into **partitions**, rebalance them with
`repartition`, and run one aggregation in parallel.

**Key idea:** 1 partition → 1 task → runs on 1 core. With 4 cores, 4 tasks run at the same time.
More partitions allow more parallel work, up to the number of cores.

**Why it matters:** this is how Spark processes data too big or too slow for one core. On a cluster, the
partitions are spread over many machines instead of many cores.

In [ ]:
print("Cores available            :", spark.sparkContext.defaultParallelism)
print("Partitions in trips_raw    :", trips_raw.rdd.getNumPartitions())

print("Rows in each partition:")
trips_raw.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

In [ ]:
# repartition() redistributes the rows evenly into N partitions (this needs a shuffle)
trips_8 = trips_raw.repartition(8)
print("Partitions after repartition(8):", trips_8.rdd.getNumPartitions())
trips_8.groupBy(F.spark_partition_id().alias("partition_id")).count().orderBy("partition_id").show()

In [ ]:
# A distributed aggregation: each task builds partial counts for its partition,
# then a shuffle sends rows with the same key to the same task for the final result.
spark.sparkContext.setJobDescription("Step 3: distributed aggregation by vendor")
trips_8.groupBy("VendorID").count().orderBy("VendorID").show()
spark.sparkContext.setJobDescription(None)

> 📸 **SCREENSHOT 3** — the partition counts above.

> 📸 **SCREENSHOT 4** — Spark UI → **Jobs** tab (shows the job for 'Step 3: distributed aggregation by vendor').

> 📸 **SCREENSHOT 5** — Spark UI → **Stages** tab (the aggregation is split into 2 stages because of the shuffle; note Shuffle Read/Write).

> 📸 **SCREENSHOT 6** — Spark UI → **Executors** tab (cores, tasks completed, memory).

## Step 4 — Clean the data (transformations and actions)

| | What it does | Examples |
|---|---|---|
| **Transformation** | Plans a new DataFrame. *Lazy*: nothing runs yet | `select`, `filter`, `withColumn`, `join`, `groupBy`, `orderBy` |
| **Action** | Runs the plan and returns or saves a result | `count`, `show`, `collect`, `toPandas`, `write` |
| *Narrow* transformation | Each partition is processed on its own; no data moves | `filter`, `select`, `withColumn` |
| *Wide* transformation | Rows must move between partitions (**shuffle**); starts a new stage | `groupBy`, `join`, `orderBy`, `dropDuplicates` |

### 4a. 🚕 Taxi trips: check data quality

Real data contains mistakes: trips with negative fares, zero distance, or a drop-off before the pickup.
We define **6 rules** a good trip must pass. The report below counts how many trips **fail** each rule.
All 6 counts are computed in **one pass** over the data, not 6 separate scans.

In [ ]:
START, END = F.lit("2024-01-01").cast("timestamp"), F.lit("2024-02-01").cast("timestamp")

prepared = (trips_raw
    .withColumn("pickup_ts",  F.col("tpep_pickup_datetime").cast("timestamp"))
    .withColumn("dropoff_ts", F.col("tpep_dropoff_datetime").cast("timestamp"))
    .withColumn("trip_duration_min",
                (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / 60)
    .withColumn("speed_mph", F.when(F.col("trip_duration_min") > 0,
                                     F.col("trip_distance") / (F.col("trip_duration_min") / 60)))
    .fillna({"passenger_count": 1}))   # missing passenger count -> assume 1 passenger

quality_rules = {
    "pickup in Jan 2024":   (F.col("pickup_ts") >= START) & (F.col("pickup_ts") < END),
    "duration 1-180 min":   F.col("trip_duration_min").between(1, 180),
    "distance 0-100 miles": (F.col("trip_distance") > 0) & (F.col("trip_distance") <= 100),
    "fare and total > 0":   (F.col("fare_amount") > 0) & (F.col("total_amount") > 0),
    "passengers 1-6":       F.col("passenger_count").between(1, 6),
    "speed <= 80 mph":      F.col("speed_mph") <= 80,
}

# Data-quality report: rows FAILING each rule (a row can fail more than one rule)
dq = prepared.agg(
    F.count("*").alias("total rows"),
    *[F.sum(F.when(F.coalesce(rule, F.lit(False)), 0).otherwise(1)).alias(name)
      for name, rule in quality_rules.items()]
).toPandas().T.rename(columns={0: "rows"})
dq

### 4b. 🚕 Build the clean taxi table, and see lazy evaluation in action

We keep only trips that pass all 6 rules, and add useful columns: hour, date, day of week, weekend flag,
tip %, and the payment method name.

**Watch the two timings:**
* **Defining** all these steps is almost instant, because Spark only writes down the plan.
* The **`count()` action** is when Spark actually reads the file and does the work.

In [ ]:
PAYMENT_NAMES = {0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge",
                 4: "Dispute", 5: "Unknown", 6: "Voided"}
payment_map = F.create_map(*[F.lit(x) for pair in PAYMENT_NAMES.items() for x in pair])
all_rules = reduce(lambda a, b: a & b, quality_rules.values())

t0 = time.perf_counter()
trips = (prepared
    .filter(all_rules)                                                     # narrow
    .withColumn("pickup_hour",    F.hour("pickup_ts"))                     # narrow
    .withColumn("pickup_date",    F.to_date("pickup_ts"))
    .withColumn("day_of_week",    F.date_format("pickup_ts", "E"))
    .withColumn("is_weekend",     F.dayofweek("pickup_ts").isin(1, 7))
    .withColumn("tip_pct",        F.round(F.col("tip_amount") / F.col("fare_amount") * 100, 2))
    .withColumn("payment_method", F.coalesce(payment_map[F.col("payment_type")], F.lit("Other")))
    .select("VendorID", "pickup_ts", "dropoff_ts", "pickup_date", "pickup_hour", "day_of_week",
            "is_weekend", "passenger_count", "trip_distance", "trip_duration_min", "speed_mph",
            "PULocationID", "DOLocationID", "payment_method", "fare_amount", "tip_amount",
            "tip_pct", "total_amount"))
t_define = time.perf_counter() - t0

t0 = time.perf_counter()
clean_rows = trips.count()                                                 # ACTION
t_action = time.perf_counter() - t0

raw_rows = int(dq.loc["total rows", "rows"])
print(f"Defining the transformations took {t_define * 1000:8.1f} ms  (lazy: nothing executed)")
print(f"The count() action took          {t_action * 1000:8.1f} ms  (whole pipeline executed)")
print(f"Rows: {raw_rows:,} raw -> {clean_rows:,} clean "
      f"({raw_rows - clean_rows:,} removed, {(raw_rows - clean_rows) / raw_rows:.1%})")
trips.show(5)

> 📸 **SCREENSHOT 7** — the data-quality report and the lazy-evaluation timings above.

### 4c. 🏛️ Clean the 311 complaints

* Remove duplicate requests (`dropDuplicates` on the request ID).
* Convert date text to timestamps with `try_cast`. A bad value becomes empty instead of crashing the job.
* Standardize borough names; anything else becomes `UNSPECIFIED`.
* Compute **how many hours each request took to close**.

The report shows how many rows were affected.

In [ ]:
c311 = (c311_raw
    .dropDuplicates(["unique_key"])                                         # wide: needs a shuffle
    .withColumn("created_ts", F.expr("try_cast(created_date AS TIMESTAMP)"))
    .withColumn("closed_ts",  F.expr("try_cast(closed_date AS TIMESTAMP)"))
    .filter(F.col("created_ts").isNotNull())
    .withColumn("borough", F.when(F.upper("borough").isin(
                    "MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"), F.upper("borough"))
                 .otherwise("UNSPECIFIED"))
    .withColumn("complaint_type", F.trim("complaint_type"))
    .withColumn("resolution_hours",
                F.when(F.col("closed_ts") >= F.col("created_ts"),
                       (F.col("closed_ts").cast("long") - F.col("created_ts").cast("long")) / 3600))
    .withColumn("created_date_only", F.to_date("created_ts"))
    .withColumn("created_hour", F.hour("created_ts"))
    .select("unique_key", "created_ts", "closed_ts", "created_date_only", "created_hour", "agency",
            "complaint_type", "descriptor", "borough", "status", "resolution_hours"))

c311_report = c311_raw.agg(
    F.count("*").alias("raw rows"),
    F.countDistinct("unique_key").alias("unique requests"),
    F.sum(F.when(F.col("closed_date").isNull(), 1).otherwise(0)).alias("not closed yet"),
    F.sum(F.when(~F.upper("borough").isin("MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND")
                 | F.col("borough").isNull(), 1).otherwise(0)).alias("borough unspecified"),
).toPandas().T.rename(columns={0: "rows"})
print(f"Clean 311 rows: {c311.count():,}")
c311_report

### 4d. 🚦 Clean the live traffic readings

* Keep only valid readings: `status = 0` and `speed > 0`. The API uses status −101 for "no data".
* Each road sensor reports many times, so a **window function** (`row_number`, newest first, per road) keeps only
  the **latest reading for each road**.

In [ ]:
traffic = (traffic_raw
    .select(F.col("link_id"), F.col("link_name"), F.upper("borough").alias("borough"),
            F.expr("try_cast(speed AS DOUBLE)").alias("speed_mph"),
            F.expr("try_cast(travel_time AS INT)").alias("travel_time_s"),
            F.expr("try_cast(data_as_of AS TIMESTAMP)").alias("reading_ts"),
            F.col("status"))
    .filter((F.col("status") == "0") & (F.col("speed_mph") > 0)))

# A sensor link reports many times; keep only its newest reading (window function)
newest_first = Window.partitionBy("link_id").orderBy(F.desc("reading_ts"))
traffic_now = (traffic.withColumn("rn", F.row_number().over(newest_first))
                      .filter(F.col("rn") == 1).drop("rn"))
print(f"Valid readings: {traffic.count():,}   Road links with a current speed: {traffic_now.count():,}")
traffic_now.agg(F.min("reading_ts").alias("oldest reading"), F.max("reading_ts").alias("newest reading")).show()

> 📸 **SCREENSHOT 8** — the 311 cleaning report and the live-traffic counts above.

## Step 4 (continued) — Answer 10 smart-city questions

**How every query works:** transformations describe the steps, then one action (`toPandas()`) runs them.
The result is small, so we show it as a Pandas table and draw a chart with matplotlib.
Each result is also saved as CSV, and each chart as PNG.

| Area | Questions |
|---|---|
| 🚕 Mobility | Q1 taxi demand by hour · Q2 busiest pickup areas · Q3 weekly pattern |
| 🏛️ Citizen services | Q4 top complaints · Q5 how fast departments respond · Q6 top problems per borough |
| 🌫️🌦️ Environment | Q7 traffic vs air pollution · Q8 cold weather vs heating complaints |
| 🚦 Live traffic | Q9 road speeds right now |
| 🏙️ All together | Q10 borough scorecard |

In [ ]:
def save_result(pdf, name):
    pdf.to_csv(f"{RESULTS}/{name}.csv", index=False)

def save_chart(fig, name):
    fig.tight_layout()
    fig.savefig(f"{CHARTS}/{name}.png", dpi=120)
    plt.show()

## 🚕 Mobility: how people move around the city

#### Q1. At what hours do people take taxis?

*Spark features: `groupBy` + `agg` (count, average)*: trips, average fare and average distance for each hour of the day.

In [ ]:
spark.sparkContext.setJobDescription("Q1: taxi demand by hour")
q1 = (trips.groupBy("pickup_hour")
           .agg(F.count("*").alias("trips"),
                F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
                F.round(F.avg("trip_distance"), 2).alias("avg_distance_mi"))
           .orderBy("pickup_hour"))
q1_pd = q1.toPandas()
save_result(q1_pd, "q1_taxi_demand_by_hour")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(q1_pd.pickup_hour, q1_pd.trips, color="#4C72B0")
ax.set(title="Q1. Taxi trips by hour of day (Jan 2024)", xlabel="Hour of day", ylabel="Trips")
ax.set_xticks(range(24))
save_chart(fig, "q1_taxi_demand_by_hour")
q1_pd

#### Q1 again, written in SQL

Spark also accepts SQL. We register the DataFrame as a table named `trips` and write the same query in SQL.
Both versions become the same execution plan, so the results are identical (checked below).

In [ ]:
trips.createOrReplaceTempView("trips")
q1_sql = spark.sql("""
    SELECT pickup_hour,
           COUNT(*)                     AS trips,
           ROUND(AVG(fare_amount), 2)   AS avg_fare,
           ROUND(AVG(trip_distance), 2) AS avg_distance_mi
    FROM trips
    GROUP BY pickup_hour
    ORDER BY pickup_hour
""")
print("SQL result identical to DataFrame result:", q1_sql.toPandas().equals(q1_pd))

#### Q2. Which areas are the busiest pickup hubs?

*Spark features: `join`, `orderBy`, `limit`.* The trips only contain zone **numbers**. Joining with the zones table
gives each trip a zone **name** and borough. Then we count pickups per zone and keep the top 10.

In [ ]:
spark.sparkContext.setJobDescription("Q2: busiest mobility hubs")
q2 = (trips.join(zones, trips.PULocationID == zones.LocationID)
           .groupBy("Zone", "Borough")
           .agg(F.count("*").alias("pickups"),
                F.round(F.sum("total_amount"), 0).alias("revenue_usd"))
           .orderBy(F.desc("pickups"))
           .limit(10))
q2_pd = q2.toPandas()
save_result(q2_pd, "q2_mobility_hubs")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(q2_pd.Zone[::-1], q2_pd.pickups[::-1] / 1e3, color="#55A868")
ax.set(title="Q2. Top 10 taxi pickup zones", xlabel="Pickups (thousands)")
save_chart(fig, "q2_mobility_hubs")
q2_pd

#### Q3. What does a typical week look like?

*Spark features: `pivot`.* `pivot` turns the hours 0–23 into columns, giving a 7 × 24 grid (day × hour).
We draw it as a heatmap: darker = more trips.

In [ ]:
spark.sparkContext.setJobDescription("Q3: weekly heatmap (pivot)")
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
q3 = trips.groupBy("day_of_week").pivot("pickup_hour", list(range(24))).count()
q3_pd = q3.toPandas().set_index("day_of_week").reindex(DAYS).fillna(0)
save_result(q3_pd.reset_index(), "q3_weekly_heatmap")

fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(q3_pd.values, aspect="auto", cmap="YlOrRd")
ax.set(title="Q3. Taxi trips by day of week and hour", xlabel="Hour of day")
ax.set_xticks(range(24)); ax.set_yticks(range(7)); ax.set_yticklabels(DAYS)
fig.colorbar(im, ax=ax, label="Trips")
save_chart(fig, "q3_weekly_heatmap")

## 🏛️ Citizen services: what residents report to the city (311)

#### Q4. What do people complain about most?

*Spark features: `groupBy`, plus a window over all rows to get each complaint type's share (%) of the total.*

In [ ]:
spark.sparkContext.setJobDescription("Q4: top 311 complaint types")
q4 = (c311.groupBy("complaint_type")
          .agg(F.count("*").alias("requests"))
          .withColumn("share_pct", F.round(F.col("requests") / F.sum("requests").over(Window.partitionBy()) * 100, 1))
          .orderBy(F.desc("requests"))
          .limit(10))
q4_pd = q4.toPandas()
save_result(q4_pd, "q4_top_complaints")

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(q4_pd.complaint_type[::-1], q4_pd.requests[::-1], color="#8172B2")
ax.set(title="Q4. Top 10 311 complaint types (Jan 2024)", xlabel="Requests")
save_chart(fig, "q4_top_complaints")
q4_pd

#### Q5. How quickly does each city department close complaints?

*Spark features: `percentile_approx` (median).* We use the **median** (the middle value) because a few very slow
cases would make the average misleading. The chart uses a log scale because times range from about an hour to many days.

In [ ]:
spark.sparkContext.setJobDescription("Q5: agency resolution time")
q5 = (c311.filter(F.col("resolution_hours").isNotNull())
          .groupBy("agency")
          .agg(F.count("*").alias("closed_requests"),
               F.round(F.percentile_approx("resolution_hours", 0.5), 1).alias("median_hours"),
               F.round(F.avg("resolution_hours"), 1).alias("avg_hours"))
          .filter(F.col("closed_requests") >= 500)
          .orderBy("median_hours"))
q5_pd = q5.toPandas()
save_result(q5_pd, "q5_agency_resolution_time")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(q5_pd.agency, q5_pd.median_hours, color="#C44E52")
ax.set(title="Q5. Median hours to close a 311 request, by agency", ylabel="Hours (median)")
ax.set_yscale("log")
save_chart(fig, "q5_agency_resolution_time")
q5_pd

#### Q6. What are the top 3 problems in each borough?

*Spark features: window function `dense_rank()`.* It ranks complaint types **inside each borough separately**,
then we keep ranks 1–3.

In [ ]:
spark.sparkContext.setJobDescription("Q6: top complaints per borough (window)")
by_borough = Window.partitionBy("borough").orderBy(F.desc("requests"))
q6 = (c311.filter(F.col("borough") != "UNSPECIFIED")
          .groupBy("borough", "complaint_type").agg(F.count("*").alias("requests"))
          .withColumn("rank_in_borough", F.dense_rank().over(by_borough))
          .filter(F.col("rank_in_borough") <= 3)
          .orderBy("borough", "rank_in_borough"))
q6_pd = q6.toPandas()
save_result(q6_pd, "q6_top_complaints_per_borough")
q6_pd

## 🌫️🌦️ Environment: air and weather

#### Q7. Do busy traffic hours have worse air?

*Spark features: a `join` of two sources on the hour, then `corr`.* We count taxi trips in each hour and match each
hour with that hour's air-quality reading.

**Correlation (r)** runs from −1 to +1:
* near **+1**: pollution goes up when traffic goes up;
* near **0**: no link.

In [ ]:
spark.sparkContext.setJobDescription("Q7: taxi activity vs air quality")
hourly_trips = (trips.groupBy(F.date_trunc("hour", "pickup_ts").alias("hour_ts"))
                     .agg(F.count("*").alias("taxi_trips")))
traffic_air = hourly_trips.join(air, "hour_ts").na.drop()

q7 = traffic_air.agg(F.round(F.corr("taxi_trips", "nitrogen_dioxide"), 3).alias("corr_trips_vs_NO2"),
                     F.round(F.corr("taxi_trips", "pm2_5"), 3).alias("corr_trips_vs_PM2_5"),
                     F.round(F.corr("taxi_trips", "carbon_monoxide"), 3).alias("corr_trips_vs_CO"))
q7_pd = q7.toPandas()
save_result(q7_pd, "q7_traffic_vs_air_quality")

by_hour = (traffic_air.groupBy(F.hour("hour_ts").alias("hour"))
                      .agg(F.avg("taxi_trips").alias("taxi_trips"), F.avg("nitrogen_dioxide").alias("no2"))
                      .orderBy("hour").toPandas())
fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(by_hour.hour, by_hour.taxi_trips, color="#4C72B0", alpha=0.6, label="Taxi trips")
ax1.set(xlabel="Hour of day", ylabel="Avg taxi trips per hour")
ax2 = ax1.twinx()
ax2.plot(by_hour.hour, by_hour.no2, "o-", color="#C44E52", label="NO₂")
ax2.set_ylabel("Avg NO₂ (µg/m³)")
ax1.set_title("Q7. Average taxi activity and NO₂ by hour of day")
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.9))
save_chart(fig, "q7_traffic_vs_air_quality")
q7_pd

#### Q8. Do cold days bring more heating complaints?

*Spark features: a `join` of 311 data with weather on the date, then `corr`.* For each day: the number of
*HEAT/HOT WATER* complaints vs the average temperature. A **negative r** means colder days bring more complaints.

In [ ]:
spark.sparkContext.setJobDescription("Q8: temperature vs heating complaints")
daily_heat = (c311.filter(F.upper("complaint_type") == "HEAT/HOT WATER")
                  .groupBy(F.col("created_date_only").alias("date"))
                  .agg(F.count("*").alias("heat_complaints")))
daily_temp = (weather.groupBy(F.to_date("hour_ts").alias("date"))
                     .agg(F.round(F.avg("temp_c"), 1).alias("avg_temp_c"),
                          F.round(F.sum("precip_mm"), 1).alias("precip_mm")))
q8 = daily_heat.join(daily_temp, "date").orderBy("date")
q8_pd = q8.toPandas()
heat_corr = q8.agg(F.corr("heat_complaints", "avg_temp_c")).first()[0] or 0.0
save_result(q8_pd, "q8_temperature_vs_heat_complaints")
print(f"Correlation between daily temperature and heating complaints: {heat_corr:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(q8_pd.avg_temp_c, q8_pd.heat_complaints, color="#DD8452")
ax.set(title=f"Q8. Daily heating complaints vs temperature (r = {heat_corr:.2f})",
       xlabel="Average daily temperature (°C)", ylabel="Heat/Hot Water complaints")
save_chart(fig, "q8_temperature_vs_heat_complaints")
q8_pd.head(10)

## 🚦 Live traffic: the city right now

#### Q9. How fast is traffic moving at this moment?

*Spark features: `groupBy` on the latest sensor readings, then `orderBy` + `limit`.* We show the average speed per
borough and the 10 slowest roads. Run the notebook again later and these numbers will change.

In [ ]:
spark.sparkContext.setJobDescription("Q9: live traffic speeds")
q9 = (traffic_now.groupBy("borough")
                 .agg(F.count("*").alias("road_links"),
                      F.round(F.avg("speed_mph"), 1).alias("avg_speed_mph"),
                      F.round(F.min("speed_mph"), 1).alias("min_speed_mph"))
                 .orderBy("avg_speed_mph"))
q9_pd = q9.toPandas()
save_result(q9_pd, "q9_live_speed_by_borough")

slowest = (traffic_now.orderBy("speed_mph")
                      .select("borough", "link_name", "speed_mph", "reading_ts").limit(10).toPandas())
save_result(slowest, "q9_slowest_roads_now")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(q9_pd.borough, q9_pd.avg_speed_mph, color="#64B5CD")
ax.set(title=f"Q9. Live average road speed by borough (snapshot {slowest.reading_ts.max()})",
       ylabel="mph")
save_chart(fig, "q9_live_speed_by_borough")
display(q9_pd)
slowest

## 🏙️ Putting it all together

#### Q10. Borough scorecard: four sources in one table

*Spark features: `createDataFrame`, several left joins, a broadcast join.* For each borough we combine:
* population (a small table we create in code, from the 2020 Census)
* taxi pickups
* 311 complaints and the median time to close them
* live road speed

We divide by population ("per 1,000 people") so large and small boroughs can be compared fairly.
**Left joins** keep every borough even if one source has no data for it.

In [ ]:
spark.sparkContext.setJobDescription("Q10: borough scorecard")
population = spark.createDataFrame(
    [("BRONX", 1472654), ("BROOKLYN", 2736074), ("MANHATTAN", 1694251),
     ("QUEENS", 2405464), ("STATEN ISLAND", 495747)],              # 2020 US Census
    ["borough", "population"])

mobility = (trips.join(F.broadcast(zones), trips.PULocationID == zones.LocationID)
                 .groupBy(F.upper("Borough").alias("borough"))
                 .agg(F.count("*").alias("taxi_pickups")))
services = (c311.groupBy("borough")
                .agg(F.count("*").alias("requests_311"),
                     F.round(F.percentile_approx("resolution_hours", 0.5), 1).alias("median_resolution_h")))
live = traffic_now.groupBy("borough").agg(F.round(F.avg("speed_mph"), 1).alias("live_speed_mph"))

q10 = (population
       .join(mobility, "borough", "left")
       .join(services, "borough", "left")
       .join(live, "borough", "left")
       .withColumn("pickups_per_1k_people", F.round(F.col("taxi_pickups") / F.col("population") * 1000, 1))
       .withColumn("requests_per_1k_people", F.round(F.col("requests_311") / F.col("population") * 1000, 1))
       .orderBy(F.desc("population")))
q10_pd = q10.toPandas()
save_result(q10_pd, "q10_borough_scorecard")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, col, title, color in zip(axes,
        ["pickups_per_1k_people", "requests_per_1k_people", "live_speed_mph"],
        ["Taxi pickups per 1k people", "311 requests per 1k people", "Live road speed (mph)"],
        ["#4C72B0", "#8172B2", "#64B5CD"]):
    ax.bar(q10_pd.borough.str.title(), q10_pd[col].fillna(0), color=color)
    ax.set_title(title); ax.tick_params(axis="x", rotation=30)
fig.suptitle("Q10. Borough smart-city scorecard")
save_chart(fig, "q10_borough_scorecard")
q10_pd

> 📸 **SCREENSHOT 9** — the query outputs and charts Q1–Q10 above.

#### How Spark runs a query: reading the execution plan

`explain()` prints the plan Spark made for Q1. Read it **from the bottom up**:

file scan → filter (our cleaning rules) → partial count per partition → **Exchange (= shuffle)** →
final count → sort.

Every `Exchange` is a shuffle, and a new stage in the Spark UI.

In [ ]:
q1.explain()
spark.sparkContext.setJobDescription(None)

## Step 5 — Make it faster (performance optimization)

**What happens here:** five experiments. Each runs the **same work twice**, once the normal way (baseline) and once
the optimized way, and times both.

**How we time fairly:**
* Each version runs **3 times** and we take the **median**, so one slow run doesn't distort the result.
* We use Spark's `noop` output. It runs the whole query and then throws the result away, so we measure the
  processing, not saving or printing.

In [ ]:
RUNS = 3
opt_results = []

def run_query(df):
    """Execute the full plan of df and discard the output."""
    df.write.format("noop").mode("overwrite").save()

def median_time(fn, runs=RUNS):
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)

def record(name, baseline_desc, baseline_s, optimized_desc, optimized_s):
    opt_results.append({"optimization": name,
                        "baseline": baseline_desc, "baseline_s": round(baseline_s, 2),
                        "optimized": optimized_desc, "optimized_s": round(optimized_s, 2),
                        "speedup_x": round(baseline_s / optimized_s, 2)})
    print(f"{name}: {baseline_desc} = {baseline_s:.2f}s  ->  {optimized_desc} = {optimized_s:.2f}s  "
          f"(speed-up {baseline_s / optimized_s:.2f}x)")

### O1. Cache: keep the cleaned data in memory

* **Problem:** a DataFrame is only a plan. Without a cache, **every** query re-reads the Parquet file and re-runs all
  the cleaning steps.
* **Fix:** `cache()` keeps the cleaned table in memory after it is computed once.
* **Expected:** later queries start from memory and run faster. The first action pays a one-time cost to fill the cache.

In [ ]:
def three_queries(df):
    run_query(df.groupBy("pickup_hour").count())
    run_query(df.groupBy("payment_method").agg(F.avg("tip_pct")))
    run_query(df.groupBy("PULocationID").agg(F.sum("total_amount")))

spark.sparkContext.setJobDescription("O1 baseline: no cache")
no_cache_s = median_time(lambda: three_queries(trips))

spark.sparkContext.setJobDescription("O1: build cache")
trips.cache()
t0 = time.perf_counter(); trips.count(); cache_build_s = time.perf_counter() - t0
print(f"One-time cost to build the cache: {cache_build_s:.2f}s")

spark.sparkContext.setJobDescription("O1 optimized: cached")
cached_s = median_time(lambda: three_queries(trips))
record("O1 Caching", "3 queries, no cache", no_cache_s, "3 queries, cached", cached_s)

> 📸 **SCREENSHOT 10** — Spark UI → **Storage** tab (shows the cached DataFrame, its size in memory, and 100% cached).

### O2. Broadcast join: send the small table to every task

* **Problem:** the default **sort-merge join** shuffles and sorts **both** tables, including all 3M trips, just to
  attach 265 zone names.
* **Fix:** `broadcast(zones)` copies the small zones table to every task, so each task joins its own trips locally.
* **Expected:** the big table is never shuffled. The plan changes from `SortMergeJoin` to `BroadcastHashJoin`.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")   # turn off automatic broadcast for the baseline

def revenue_by_borough(zone_df):
    return (trips.join(zone_df, trips.PULocationID == zone_df.LocationID)
                 .groupBy("Borough").agg(F.sum("total_amount")))

smj = revenue_by_borough(zones)
bhj = revenue_by_borough(F.broadcast(zones))                     # broadcast hint

print("Baseline plan  :", "SortMergeJoin" if "SortMergeJoin" in smj._jdf.queryExecution().toString() else "?")
print("Optimized plan :", "BroadcastHashJoin" if "BroadcastHashJoin" in bhj._jdf.queryExecution().toString() else "?")
bhj.explain()

spark.sparkContext.setJobDescription("O2 baseline: sort-merge join")
smj_s = median_time(lambda: run_query(smj))
spark.sparkContext.setJobDescription("O2 optimized: broadcast join")
bhj_s = median_time(lambda: run_query(bhj))
record("O2 Broadcast join", "sort-merge join", smj_s, "broadcast hash join", bhj_s)

spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # restore the default (10 MB)

> 📸 **SCREENSHOT 11** — Spark UI → **SQL / DataFrame** tab → open the 'O2 optimized: broadcast join' query (diagram shows *BroadcastHashJoin*), and the 'O2 baseline' one (shows *SortMergeJoin* + 2 Exchanges).

### O3 and O4. The right number of shuffle partitions, and letting AQE handle it

* **Problem:** after a shuffle, Spark splits the data into **200 partitions** by default. That suits big clusters,
  but with a few cores and a few million rows, 200 tiny tasks mostly waste time on scheduling.
* **O3 fix:** set the number by hand to 2 × the number of cores. AQE is switched off here so we see the raw effect.
* **O4 fix:** keep 200, but turn on **Adaptive Query Execution (AQE)**. AQE looks at the real data size while the
  query runs and merges small partitions by itself.

In [ ]:
def zone_pairs():
    return (trips.groupBy("PULocationID", "DOLocationID")
                 .agg(F.count("*").alias("trips"), F.avg("fare_amount").alias("avg_fare")))

spark.conf.set("spark.sql.adaptive.enabled", "false")

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.sparkContext.setJobDescription("O3 baseline: 200 shuffle partitions, AQE off")
p200_s = median_time(lambda: run_query(zone_pairs()))

tuned = str(spark.sparkContext.defaultParallelism * 2)
spark.conf.set("spark.sql.shuffle.partitions", tuned)
spark.sparkContext.setJobDescription(f"O3 optimized: {tuned} shuffle partitions, AQE off")
tuned_s = median_time(lambda: run_query(zone_pairs()))
record("O3 Shuffle partitions", "200 partitions (default)", p200_s, f"{tuned} partitions (2 x cores)", tuned_s)

spark.conf.set("spark.sql.shuffle.partitions", "200")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.sparkContext.setJobDescription("O4 optimized: 200 partitions, AQE on")
aqe_s = median_time(lambda: run_query(zone_pairs()))
record("O4 AQE", "200 partitions, AQE off", p200_s, "200 partitions, AQE on", aqe_s)

### O5. File format: CSV vs Parquet

We save the same taxi data as CSV once, then run the same query on both files.

| CSV | Parquet |
|---|---|
| Plain text; every value must be parsed | Binary, typed, compressed |
| Must read **every column** | Reads **only the columns the query needs** (*column pruning*) |
| Must read every row | Can skip blocks of rows that cannot match the filter (*predicate pushdown*) |

In [ ]:
CSV_PATH = f"{PROCESSED}/trips_csv"
spark.sparkContext.setJobDescription("O5: write CSV copy")
if not os.path.exists(CSV_PATH):
    trips_raw.write.mode("overwrite").option("header", True).csv(CSV_PATH)

def avg_fare_by_payment(df):
    return df.filter(F.col("fare_amount") > 0).groupBy("payment_type").agg(F.avg("fare_amount"))

csv_df = spark.read.option("header", True).schema(trips_raw.schema).csv(CSV_PATH)
parquet_df = spark.read.parquet(TRIPS_PATH)

spark.sparkContext.setJobDescription("O5 baseline: CSV")
csv_s = median_time(lambda: run_query(avg_fare_by_payment(csv_df)))
spark.sparkContext.setJobDescription("O5 optimized: Parquet")
parquet_s = median_time(lambda: run_query(avg_fare_by_payment(parquet_df)))
record("O5 File format", "CSV", csv_s, "Parquet", parquet_s)

print(f"\nStorage size: CSV {size_mb(CSV_PATH):.0f} MB  vs  Parquet {size_mb(TRIPS_PATH):.0f} MB")
print("\nParquet scan reads only these columns (column pruning) and pushes the filter down:")
plan = avg_fare_by_payment(parquet_df)._jdf.queryExecution().executedPlan().toString()
print("\n".join(line.strip() for line in plan.splitlines() if "FileScan" in line))
spark.sparkContext.setJobDescription(None)

### Results of all five experiments

In [ ]:
opt_df = pd.DataFrame(opt_results)
save_result(opt_df, "optimization_results")
opt_df

> 📸 **SCREENSHOT 12** — the optimization summary table above.

## Step 6 — Save the results

We save the cleaned taxi data as **Parquet**, the format any later job should read.
We also list everything saved so far: each query's result as CSV and each chart as PNG.

In [ ]:
spark.sparkContext.setJobDescription("Step 6: write cleaned trips")
CLEAN_PATH = f"{PROCESSED}/trips_clean_parquet"
trips.write.mode("overwrite").parquet(CLEAN_PATH)
print(f"Cleaned trips saved to {CLEAN_PATH} ({size_mb(CLEAN_PATH):.0f} MB, "
      f"{spark.read.parquet(CLEAN_PATH).count():,} rows)")

for folder in (RESULTS, CHARTS):
    print(f"\n{folder}/")
    for name in sorted(os.listdir(folder)):
        print("   ", name)
spark.sparkContext.setJobDescription(None)

## Step 7 — Analyze performance

### 7a. Which optimization helped most?

Red = baseline, green = optimized. The number next to each pair is how many times faster the optimized version is.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
y = range(len(opt_df))
ax.barh([i + 0.2 for i in y], opt_df.baseline_s, height=0.4, label="Baseline", color="#C44E52")
ax.barh([i - 0.2 for i in y], opt_df.optimized_s, height=0.4, label="Optimized", color="#55A868")
ax.set_yticks(list(y)); ax.set_yticklabels(opt_df.optimization); ax.invert_yaxis()
for i, row in opt_df.iterrows():
    ax.text(max(row.baseline_s, row.optimized_s) * 1.02, i, f"{row.speedup_x}x", va="center")
ax.set(title="Baseline vs optimized (median of 3 runs)", xlabel="Seconds (lower is better)")
ax.set_xlim(0, opt_df[["baseline_s", "optimized_s"]].max().max() * 1.15)
ax.legend()
save_chart(fig, "7a_optimization_speedup")

### 7b. Pandas vs Spark: which is faster, and when?

The same small job in both tools (read the file, filter, count trips and average fare per hour), on 100k, 500k,
1M and ~3M rows.

* **Pandas** runs on one core and must load all the data into memory, but it starts instantly.
* **Spark** pays a small fixed cost to plan and schedule tasks, then uses all cores.

So expect Pandas to win on small data, and Spark to catch up as the data grows.

In [ ]:
total_rows = trips_raw.count()
sizes = [n for n in (100_000, 500_000, 1_000_000) if n < total_rows] + [total_rows]

spark.sparkContext.setJobDescription("7b: write samples")
sample_paths = {}
for n in sizes:
    path = TRIPS_PATH if n == total_rows else f"{PROCESSED}/sample_{n}"
    if n != total_rows and not os.path.exists(path):
        trips_raw.limit(n).write.mode("overwrite").parquet(path)
    sample_paths[n] = path

def pandas_job(path):
    pdf = pd.read_parquet(path, columns=["tpep_pickup_datetime", "fare_amount"])
    pdf = pdf[pdf.fare_amount > 0]
    return pdf.groupby(pdf.tpep_pickup_datetime.dt.hour)["fare_amount"].agg(["count", "mean"])

def spark_job(path):
    df = spark.read.parquet(path).filter(F.col("fare_amount") > 0)
    return (df.groupBy(F.hour(F.col("tpep_pickup_datetime").cast("timestamp")).alias("hour"))
              .agg(F.count("*"), F.avg("fare_amount")).collect())

spark.sparkContext.setJobDescription("7b: Spark vs Pandas")
comparison = []
for n, path in sample_paths.items():
    comparison.append({"rows": n,
                       "pandas_s": round(median_time(lambda: pandas_job(path)), 3),
                       "spark_s":  round(median_time(lambda: spark_job(path)), 3)})
cmp_df = pd.DataFrame(comparison)
save_result(cmp_df, "pandas_vs_spark")

full_pdf_mb = pd.read_parquet(TRIPS_PATH).memory_usage(deep=True).sum() / 1e6
print(f"Pandas needs {full_pdf_mb:,.0f} MB of RAM to hold the full month in memory "
      f"(the file on disk is {size_mb(TRIPS_PATH):.0f} MB).")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cmp_df.rows / 1e6, cmp_df.pandas_s, "o-", label="Pandas", color="#DD8452")
ax.plot(cmp_df.rows / 1e6, cmp_df.spark_s, "o-", label="Spark", color="#4C72B0")
ax.set(title="Pandas vs Spark: same job, growing data", xlabel="Rows (millions)", ylabel="Seconds")
ax.legend()
save_chart(fig, "7b_pandas_vs_spark")
spark.sparkContext.setJobDescription(None)
cmp_df

> 📸 **SCREENSHOT 13** — the Pandas vs Spark table and chart.

### 7c. Does adding CPU cores make Spark faster?

The same pipeline (read → clean → join → aggregate) runs with 1 core (`local[1]`), 2 cores, and all cores
(`local[*]`). Spark is restarted for each setting. This is our small-scale version of "comparing cluster modes".

> ⚠️ **Take all the Spark UI screenshots (4, 5, 6, 10, 11) BEFORE running the next cell.** Restarting Spark clears the UI.

In [ ]:
def pipeline_workload(session):
    raw = session.read.parquet(TRIPS_PATH)
    zone_df = session.read.csv(ZONES_PATH, header=True, schema=zone_schema)
    cleaned = (raw.withColumn("pickup_ts", F.col("tpep_pickup_datetime").cast("timestamp"))
                  .filter((F.col("fare_amount") > 0) & (F.col("trip_distance") > 0)))
    result = (cleaned.join(F.broadcast(zone_df), cleaned.PULocationID == zone_df.LocationID)
                     .groupBy("Borough", F.hour("pickup_ts").alias("hour"))
                     .agg(F.count("*"), F.avg("fare_amount"), F.sum("total_amount")))
    result.write.format("noop").mode("overwrite").save()

cpu_count = os.cpu_count() or 1
modes = ["local[1]"] + (["local[2]"] if cpu_count > 2 else []) + ["local[*]"]

scaling = []
for mode in modes:
    spark.stop()
    spark = create_spark(mode)
    spark.sparkContext.setLogLevel("ERROR")
    cores = spark.sparkContext.defaultParallelism
    seconds = median_time(lambda: pipeline_workload(spark))
    scaling.append({"mode": mode, "cores": cores, "seconds": round(seconds, 2)})
    print(f"{mode:9s} ({cores} cores): {seconds:.2f}s")

scale_df = pd.DataFrame(scaling)
scale_df["speedup_vs_1_core"] = (scale_df.seconds.iloc[0] / scale_df.seconds).round(2)
save_result(scale_df, "scaling_by_cores")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar([f"{m}\n{c} cores" for m, c in zip(scale_df["mode"], scale_df.cores)], scale_df.seconds, color="#4C72B0")
ax.set(title="Same pipeline, different number of cores", ylabel="Seconds (lower is better)")
save_chart(fig, "7c_scaling_by_cores")
scale_df

> 📸 **SCREENSHOT 14** — the scaling table and chart.

### 7d. What we learned about the city (smart city insights)

In [ ]:
peak = q1_pd.loc[q1_pd.trips.idxmax()]
top_hub = q2_pd.iloc[0]
top_complaint = q4_pd.iloc[0]
fastest_agency, slowest_agency = q5_pd.iloc[0], q5_pd.iloc[-1]
q7_row = q7_pd.iloc[0]

print(f"🚕 Peak taxi hour: {int(peak.pickup_hour)}:00 with {int(peak.trips):,} trips in the month; "
      f"busiest hub: {top_hub.Zone} ({int(top_hub.pickups):,} pickups).")
print(f"🏛️ Most common 311 complaint: {top_complaint.complaint_type} ({top_complaint.share_pct}% of requests).")
print(f"🏛️ Fastest agency: {fastest_agency.agency} (median {fastest_agency.median_hours} h); "
      f"slowest: {slowest_agency.agency} (median {slowest_agency.median_hours} h).")
print(f"🌫️ Correlation of hourly taxi trips with NO₂: {q7_row.corr_trips_vs_NO2}, with PM2.5: {q7_row.corr_trips_vs_PM2_5}.")
print(f"🌦️ Correlation of daily temperature with heating complaints: {heat_corr:.2f} "
      f"({'colder days bring more complaints' if heat_corr < 0 else 'no clear cold-weather effect'}).")
if len(q9_pd):
    print(f"🚦 Live: slowest borough right now is {q9_pd.iloc[0].borough.title()} "
          f"({q9_pd.iloc[0].avg_speed_mph} mph); slowest road: {slowest.iloc[0].link_name} ({slowest.iloc[0].speed_mph} mph).")

### 7e. All measured numbers in one place

In [ ]:
print("OPTIMIZATIONS")
for r in opt_results:
    print(f"  {r['optimization']:24s} {r['baseline_s']:7.2f}s -> {r['optimized_s']:7.2f}s   {r['speedup_x']}x faster")

print("\nPANDAS vs SPARK")
for r in comparison:
    winner = "Pandas" if r["pandas_s"] < r["spark_s"] else "Spark"
    print(f"  {r['rows']:>10,} rows: Pandas {r['pandas_s']:6.2f}s | Spark {r['spark_s']:6.2f}s  -> {winner} faster")

print("\nSCALING BY CORES")
for r in scale_df.to_dict("records"):
    print(f"  {r['mode']:9s} {r['cores']} cores: {r['seconds']:6.2f}s  ({r['speedup_vs_1_core']}x vs 1 core)")

### Conclusions

**About performance**

1. **Parquet instead of CSV** gave the largest speed-up in our tests, and the file is about 6x smaller.
   Columnar storage means Spark reads only the columns a query needs.
2. **Caching** pays off whenever the same cleaned data feeds several queries. Without it, Spark redoes the reading
   and cleaning for every action, because a DataFrame is only a plan.
3. **Broadcast joins** avoid shuffling the big table when the other table is small. Spark does this automatically
   for tables under 10 MB; `broadcast()` makes it explicit.
4. **Shuffle partitions:** the default of 200 is too many for a few cores. Tuning it by hand or enabling **AQE**
   (on by default in Spark 3.2+) removes most of that overhead.
5. **Pandas vs Spark:** Pandas is faster on small data because Spark has a start-up cost. As data grows Spark catches
   up, and unlike Pandas it is not limited to one machine's memory.
6. **More cores help, but not perfectly:** part of every job (reading the file, planning) does not get faster with
   more cores. On a real cluster the same code spreads over many machines.

**About the smart city**

7. One engine and one API handled six sources in four formats (Parquet, CSV, JSON, live API) and joined them by
   **time** (hour, day) and **place** (borough) to answer questions that need more than one dataset.
8. Re-running the notebook pulls a new live-traffic snapshot. That is the first step towards a real-time city
   dashboard; Spark Structured Streaming would make it continuous.

In [ ]:
# Zip all charts and result tables (downloads automatically in Colab)
archive = shutil.make_archive("case_study_outputs", "zip", "outputs")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Saved", archive)